# Stage 4-3. Experiment, Predictor, Checkpoint

이 노트북은 `src/core/experiment.py`, `src/core/predictor.py`, `src/core/checkpoints.py`를 실습한다.

| 클래스/함수 | 역할 |
|---|---|
| `Experiment` | config 기반으로 dataset, dataloader, model, optimizer, trainer, evaluator, predictor를 조립하는 최상위 진입점 |
| `Experiment.run()` | `num_epochs` 동안 train/test 루프 실행 → epoch별 log 목록 반환 |
| `Predictor.predict(images)` | raw logits → task별 후처리 예측 `{logits, predictions}` 반환 |
| `checkpoints.save(model, path)` | model.params를 `.npz` 파일로 저장 |
| `checkpoints.load(model, path)` | `.npz` 파일에서 model.params를 in-place 복원 |

**학습 목표**
1. `Experiment(config)`로 전체 파이프라인을 한 번에 조립하는 패턴을 익힌다.
2. `Experiment.run()`의 반환값 구조를 확인한다.
3. `Predictor`의 task별 후처리(argmax/threshold/round_clip)를 비교한다.
4. checkpoint 저장/불러오기 후 예측 결과가 동일함을 검증한다.
5. 학습 곡선과 예측 grid를 시각화한다.

## 0. 환경 설정

In [ ]:
import sys
sys.path.insert(0, "../..")

import numpy as np
import matplotlib.pyplot as plt

from src.core.experiment import Experiment
from src.core.predictor import Predictor
from src.core import checkpoints
from src.data.mnist import MnistDataset
from src.task import get_task_spec

## 1. Experiment — config 구성

In [ ]:
config = {
    "dataset_dir": "/mnt/d/datasets/mnist",
    "task":        "multiclass",
    "model":       "mlp",
    "seed":        42,
    "batch_size":  64,
    "num_epochs":  5,
    "lr":          0.01,
}

exp = Experiment(config)

print("Experiment 구성 요소:")
print(f"  train_loader : {len(exp.train_loader)} 배치")
print(f"  test_loader  : {len(exp.test_loader)} 배치")
print(f"  model        : {type(exp.model).__name__}")
print(f"  trainer      : {type(exp.trainer).__name__}")
print(f"  evaluator    : {type(exp.evaluator).__name__}")
print(f"  predictor    : {type(exp.predictor).__name__}")

## 2. Experiment.run() — 전체 학습 실행

In [ ]:
logs = exp.run()

print("run() 반환값 구조 (첫 번째 epoch):")
print(logs[0])
print()
print(f"{'epoch':>5} | {'train loss':>10} {'train acc':>10} | {'test loss':>10} {'test acc':>10}")
print("-" * 55)
for log in logs:
    print(f"{log['epoch']:>5} | "
          f"{log['train']['loss']:>10.4f} {log['train']['metric']:>10.4f} | "
          f"{log['test']['loss']:>10.4f} {log['test']['metric']:>10.4f}")

## 3. 학습 곡선 시각화

In [ ]:
epochs     = [l["epoch"]           for l in logs]
train_loss = [l["train"]["loss"]   for l in logs]
test_loss  = [l["test"]["loss"]    for l in logs]
train_acc  = [l["train"]["metric"] for l in logs]
test_acc   = [l["test"]["metric"]  for l in logs]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle("Experiment.run() — Multiclass MLP", fontsize=13, fontweight="bold")

ax1.plot(epochs, train_loss, label="train", color="steelblue", linewidth=2, marker="o")
ax1.plot(epochs, test_loss,  label="test",  color="#E87B4C",   linewidth=2, marker="o", linestyle="--")
ax1.set_title("Cross-Entropy Loss")
ax1.set_xlabel("epoch")
ax1.set_ylabel("loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, train_acc, label="train", color="steelblue", linewidth=2, marker="o")
ax2.plot(epochs, test_acc,  label="test",  color="#E87B4C",   linewidth=2, marker="o", linestyle="--")
ax2.set_title("Accuracy")
ax2.set_xlabel("epoch")
ax2.set_ylabel("accuracy")
ax2.set_ylim(0, 1)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Predictor — task별 후처리 비교

In [ ]:
DATASET_DIR = "/mnt/d/datasets/mnist"

for task, mode in [("multiclass", "argmax"), ("binary", "threshold"), ("regression", "round_clip")]:
    from src.models.mlp import MLP
    spec = get_task_spec(task)
    mdl  = MLP(task=task, seed=42)
    prd  = Predictor(mdl, spec)

    ds   = MnistDataset("test", task, dataset_dir=DATASET_DIR)
    imgs = np.stack([ds[i][0] for i in range(8)])

    result = prd.predict(imgs)
    print(f"task={task} (mode={mode})")
    print(f"  logits shape     : {result['logits'].shape}")
    print(f"  predictions shape: {result['predictions'].shape}")
    print(f"  predictions      : {result['predictions']}")
    print()

## 5. Predictor — 학습 후 예측 grid

In [ ]:
# 학습된 exp.model로 예측
test_ds  = MnistDataset("test", "multiclass", dataset_dir=DATASET_DIR)
N_SHOW   = 16
imgs_show = np.stack([test_ds[i][0] for i in range(N_SHOW)])
true_lbls = np.array([test_ds[i][1].argmax() for i in range(N_SHOW)])

result = exp.predictor.predict(imgs_show)
preds  = result["predictions"]

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
fig.suptitle("Predictor — Multiclass (after 5 epochs)", fontsize=12, fontweight="bold")

for i, ax in enumerate(axes.ravel()):
    ax.imshow(imgs_show[i].reshape(28, 28), cmap="gray")
    color = "green" if true_lbls[i] == preds[i] else "red"
    ax.set_title(f"T:{true_lbls[i]} P:{preds[i]}", fontsize=8, color=color)
    ax.axis("off")

plt.tight_layout()
plt.show()

correct = (true_lbls == preds).sum()
print(f"정답 {correct}/{N_SHOW} = {correct/N_SHOW:.0%}")

## 6. Checkpoint — 저장 / 불러오기 검증

In [ ]:
import tempfile, os

# 저장 전 예측
imgs_ck = np.stack([test_ds[i][0] for i in range(4)])
pred_before = exp.predictor.predict(imgs_ck)["predictions"]

# 임시 파일에 저장
with tempfile.NamedTemporaryFile(suffix=".npz", delete=False) as f:
    ck_path = f.name

checkpoints.save(exp.model, ck_path.replace(".npz", ""))
print(f"저장 경로: {ck_path}")
print(f"파일 크기: {os.path.getsize(ck_path) / 1024:.1f} KB")

# 파라미터를 0으로 초기화
from src.models.mlp import MLP as _MLP
model_new = _MLP(task="multiclass", seed=0)
for p in model_new.params:
    p[...] = 0.0

pred_zero = Predictor(model_new, get_task_spec("multiclass")).predict(imgs_ck)["predictions"]
print(f"\n0으로 초기화 후 예측   : {pred_zero}")

# checkpoint 불러오기
checkpoints.load(model_new, ck_path.replace(".npz", ""))
pred_loaded = Predictor(model_new, get_task_spec("multiclass")).predict(imgs_ck)["predictions"]
print(f"checkpoint 복원 후 예측: {pred_loaded}")
print(f"저장 전 예측           : {pred_before}")
print(f"\n복원 성공: {np.array_equal(pred_before, pred_loaded)}")

os.unlink(ck_path)

## 7. 저장된 .npz 파일 구조 확인

In [ ]:
import tempfile

with tempfile.NamedTemporaryFile(suffix=".npz", delete=False) as f:
    path_inspect = f.name

checkpoints.save(exp.model, path_inspect.replace(".npz", ""))
data = np.load(path_inspect)

print(".npz 파일 내 배열 목록:")
total = 0
for key in sorted(data.files):
    arr = data[key]
    print(f"  {key}: shape={arr.shape}, dtype={arr.dtype}, size={arr.size:,}")
    total += arr.size
print(f"  ─────────────────────────")
print(f"  총 파라미터 수: {total:,}")

os.unlink(path_inspect)

## 8. 정리

**Experiment 조립 구조**
```
Experiment(config)
  ├─ MnistDataset(train) + MnistDataset(test)
  ├─ DataLoader(train) + DataLoader(test)
  ├─ MLP(task, seed)  ← 또는 CNN
  ├─ SGD(model, lr)
  ├─ Trainer(model, optimizer, task_spec)
  ├─ Evaluator(model, task_spec)
  └─ Predictor(model, task_spec)
```

**Predictor task별 후처리**

| task | prediction_mode | 후처리 |
|---|---|---|
| multiclass | argmax | `logits.argmax(axis=1)` |
| binary | threshold | `sigmoid(logits) >= 0.5` |
| regression | round_clip | `round(clip(logits*9, 0, 9))` |

**Checkpoint 인터페이스**
```python
checkpoints.save(model, path)   # param_0, param_1, ... 키로 저장
checkpoints.load(model, path)   # in-place 복원
```

**핵심 설계 원칙**
- `Experiment`는 의존성 주입(DI) 패턴으로 내부 구현을 숨기고 config 하나로 전체 파이프라인을 조립한다.
- `Predictor`는 task_spec의 `prediction_mode`로 분기하므로 task가 바뀌어도 동일 인터페이스를 유지한다.
- Stage 5의 CLI 스크립트는 `Experiment`를 조립 진입점으로 사용한다.